In [ ]:
import requests
import pandas as pd
import numpy as np
from collections import Counter

# ============================================
# ADIM 1:  Veri Çekme
# ============================================

url = "https://api.platform.opentargets.org/api/v4/graphql"



all_rows = []
for page_idx in range(0, 3):
    query = f"""
    query {{
      disease(efoId: "MONDO_0004975") {{
        associatedTargets(page: {{index: {page_idx}, size: 1000}}) {{
          rows {{
            target {{
              id
              approvedSymbol
              approvedName
              biotype
              proteinIds {{
                id
                source
              }}
            }}
            score
            datatypeScores {{
              id
              score
            }}
          }}
        }}
      }}
    }}
    """
    response = requests.post(url, json={"query": query})
    data = response.json()
    rows = data["data"]["disease"]["associatedTargets"]["rows"]
    if not rows:
        break
    all_rows.extend(rows)

print(f"✓ Toplam {len(all_rows)} gen bulundu")


records = []
for r in all_rows:
     datatype_dict = {d["id"]: d["score"] for d in r["datatypeScores"]}

    # UniProt ID'yi bul
    uniprot_id = None
    if r["target"].get("proteinIds"):
        for pid in r["target"]["proteinIds"]:
            if pid["source"] == "uniprot_swissprot":
                uniprot_id = pid["id"]
                break

    records.append({
        "target_id": r["target"]["id"],
        "gene_symbol": r["target"]["approvedSymbol"],
        "gene_name": r["target"].get("approvedName", ""),
        "biotype": r["target"].get("biotype", ""),
        "uniprot_id": uniprot_id,
        "overall_score": r["score"],
        "genetic_association": datatype_dict.get("genetic_association", 0),
        "somatic_mutation": datatype_dict.get("somatic_mutation", 0),
        "literature": datatype_dict.get("literature", 0),
        "rna_expression": datatype_dict.get("rna_expression", 0),
        "animal_model": datatype_dict.get("animal_model", 0),
        "affected_pathway": datatype_dict.get("affected_pathway", 0)
    })

df = pd.DataFrame(records)
print(f"✓ DataFrame oluşturuldu: {df.shape}")

In [ ]:
df

In [ ]:
# ============================================
# ADIM 3: Alzheimer'a Özel Filtreleme
# ============================================


# Alzheimer için daha uygun kriter: Genetik İLİŞKİ + (Literature VEYA RNA Expression)
df["alzheimer_score"] = (
    df["genetic_association"] * 0.4 +
    df["literature"] * 0.3 +
    df["rna_expression"] * 0.2 +
    df["animal_model"] * 0.1
)

# Sadece protein-coding genleri alınıyor ilaç hedefi olduğu için
df_protein = df[df["biotype"] == "protein_coding"].copy()
print(f"✓ Protein-coding genler: {len(df_protein)}")

# Eşik değerleri
filtered = df_protein[
    (df_protein["alzheimer_score"] >= 0.20) &  # Daha esnek eşik
    (df_protein["uniprot_id"].notna())  # UniProt ID olması şart chemble için
].copy()


print(f"✓ Filtreleme sonrası: {len(filtered)} gen")
print(f"\nİlk 15 gen:")
print(filtered[["gene_symbol", "overall_score", "alzheimer_score"]].head(15))

In [ ]:
# ============================================
# ADIM 4: Protein Sınıfı Bilgisi Ekleme
# ============================================
import requests
import pandas as pd
import time

def get_protein_class(target_id):
    query = """
    query ($id: String!) {
      target(ensemblId: $id) {
        approvedSymbol
        targetClass {
          label
          level
        }
      }
    }
    """
    try:
        response = requests.post(url, json={"query": query, "variables": {"id": target_id}})
        data = response.json()
        symbol = data.get("data", {}).get("target", {}).get("approvedSymbol")
        classes = data.get("data", {}).get("target", {}).get("targetClass", [])
        if not symbol:
            return {"gene_symbol": target_id, "protein_class": None}
        if not classes:
            return {"gene_symbol": symbol, "protein_class": None}
        class_list = [c["label"] for c in classes]
        return {"gene_symbol": symbol, "protein_class": ", ".join(class_list)}
    except Exception as e:
        print("Hata:", e)
        return {"gene_symbol": target_id, "protein_class": None}

# Örnek gen listesi
genes = ["ENSG00000087088", "ENSG00000182185", "ENSG00000141510"]

results = []
for g in genes:
    results.append(get_protein_class(g))
    time.sleep(0.2)  # API rate limit'e takılmamak için küçük gecikme

df_proteins = pd.DataFrame(results)
print(df_proteins)


In [ ]:
import requests

def get_combined_protein_info(target_id, uniprot_id):
    """
    Open Targets + UniProt'tan protein sınıfı/fonksiyonu alır.
    1. Open Targets API'den targetClass bilgisi alınır.
    2. Eğer bulunamazsa UniProt'tan protein fonksiyon açıklaması döner.
    """
    # 1️⃣ Open Targets sorgusu
    ot_query = f"""
    query {{
      target(ensemblId: "{target_id}") {{
        targetClass {{
          label
          level
        }}
      }}
    }}
    """
    try:
        ot_response = requests.post(
            "https://api.platform.opentargets.org/api/v4/graphql",
            json={"query": ot_query},
            timeout=10
        )
        ot_data = ot_response.json()
        classes = ot_data.get("data", {}).get("target", {}).get("targetClass", [])
        if classes:
            level1 = [c["label"] for c in classes if c.get("level") == 1]
            if level1:
                return level1[0]
    except Exception as e:
        pass  # Hata olursa UniProt'a geç

    # 2️⃣ UniProt sorgusu (yedek)
    if uniprot_id:
        try:
            uni_url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
            uni_response = requests.get(uni_url, timeout=10)
            if uni_response.status_code == 200:
                uni_data = uni_response.json()
                protein_desc = uni_data.get("proteinDescription", {})
                name_block = protein_desc.get("recommendedName", {})
                full_name = name_block.get("fullName", {}).get("value")
                if full_name:
                    return full_name
        except Exception as e:
            pass

    return None


In [ ]:
print("🔎 Open Targets + UniProt protein sınıfları:")

sample_genes = filtered.head(15).copy()
sample_genes["protein_class"] = sample_genes.apply(
    lambda x: get_combined_protein_info(x["target_id"], x["uniprot_id"]),
    axis=1
)

print(sample_genes[["gene_symbol", "protein_class"]])

In [ ]:

# ============================================
# ADIM 5: Alzheimer'a Özel Öncelikli Genler
# ============================================

# Bilinen Alzheimer gen aileleri
KINASES = ["GSK3B", "CDK5", "MAPK1", "MAPK3", "AKT1", "PRKACA", "DYRK1A", "ROCK1"]
PROTEASES = ["BACE1", "ADAM10", "PSEN1", "PSEN2", "CAPN1", "CTSB", "CTSD"]
GPCR = ["CHRM1", "CHRM3", "DRD1", "DRD2", "ADRB2", "HTR2A", "HTR6"]
OXIDOREDUCTASES = ["SOD1", "SOD2", "CAT", "GPX1", "NOS1", "NOS3"]
TRANSFERASES = ["CHAT", "ACHE", "BCHE", "COMT", "PNMT"]
METAL_BINDING = ["APP", "PRNP", "FTL", "FTH1", "MT1A", "MT2A"]

priority_genes = {
    "Kinase": KINASES,
    "Protease": PROTEASES,
    "GPCR": GPCR,
    "Oxidoreductase": OXIDOREDUCTASES,
    "Transferase": TRANSFERASES,
    "Metal-binding": METAL_BINDING
}

# Sınıflandırma
def classify_gene(symbol):
    for class_name, gene_list in priority_genes.items():
        if symbol in gene_list:
            return class_name
    return "Other"

filtered["manual_class"] = filtered["gene_symbol"].apply(classify_gene)

# Öncelikli genleri ayır
priority_df = filtered[filtered["manual_class"] != "Other"].copy()
print(f"\n✓ Öncelikli sınıflara ait genler: {len(priority_df)}")
print("\nSınıf dağılımı:")
print(priority_df["manual_class"].value_counts())


In [ ]:
print("\n" + "=" * 60)
print("ADIM 6: TÜM PROTEIN SINIFLARI İÇİN DETAYLI ANALİZ")
print("=" * 60)

# Her sınıf için istatistikler
print("\n📊 SINIF BAZLI İSTATİSTİKLER:\n")
for class_name in priority_df["manual_class"].unique():
    class_genes = priority_df[priority_df["manual_class"] == class_name]
    print(f"\n🔹 {class_name} ({len(class_genes)} gen)")
    print(f"   Ortalama Alzheimer skoru: {class_genes['alzheimer_score'].mean():.3f}")
    print(f"   En yüksek skorlu 3 gen:")
    top3 = class_genes.nlargest(3, "alzheimer_score")[["gene_symbol", "alzheimer_score"]]
    for _, row in top3.iterrows():
        print(f"      • {row['gene_symbol']}: {row['alzheimer_score']:.3f}")

# Sınıfları ChEMBL veri potansiyeline göre sırala
class_priority = {
    "Kinase": {"priority": 1, "chembl_data": "Çok Yüksek (30k+)"},
    "GPCR": {"priority": 2, "chembl_data": "Yüksek (15k+)"},
    "Protease": {"priority": 3, "chembl_data": "Orta (8k+)"},
    "Oxidoreductase": {"priority": 4, "chembl_data": "Düşük (2k+)"},
    "Transferase": {"priority": 5, "chembl_data": "Düşük (3k+)"},
    "Metal-binding": {"priority": 6, "chembl_data": "Çok Düşük (<1k)"}
}



In [ ]:
# Tüm filtrelenmiş genler
filtered.to_csv("alzheimer_all_filtered_genes.csv", index=False)
print(f"✓ Tüm genler kaydedildi: alzheimer_all_filtered_genes.csv ({len(filtered)} gen)")

# Öncelikli genler
priority_df.to_csv("alzheimer_priority_genes.csv", index=False)
print(f"✓ Öncelikli genler kaydedildi: alzheimer_priority_genes.csv ({len(priority_df)} gen)")


In [ ]:

targets2 = pd.read_csv("alzheimer_priority_genes.csv")


In [ ]:


"""
EN İYİ 15 GENİ SEÇ
"""

import pandas as pd
import numpy as np

print("=" * 80)
print("ADIM 1: EN YÜKSEK SKORLU 15 GENİ SEÇİYORUZ")
print("=" * 80)

# Öncelikli genler
try:
    priority_df = pd.read_csv("alzheimer_priority_genes.csv")
    print(f"✓ Dosya yüklendi: {len(priority_df)} öncelikli gen")
except FileNotFoundError:
    print("HATA: 'alzheimer_priority_genes.csv' bulunamadı!")

    exit()

# ============================================
# YÖNTEM 1: Alzheimer Skoruna Göre
# ============================================
print("\n" + "=" * 80)
print("YÖNTEM 1: Alzheimer Skoruna Göre Top 15")
print("=" * 80)

top15_by_score = priority_df.nlargest(15, "alzheimer_score").copy()

print("\nSeçilen 15 gen:")
print(top15_by_score[["gene_symbol", "manual_class", "alzheimer_score", "overall_score"]].to_string(index=False))

print("\n Sınıf Dağılımı:")
print(top15_by_score["manual_class"].value_counts())







final_selection = top15_by_score.copy()

print(f"\n {len(final_selection)} gen seçildi")
print("\nSeçilen genler:")
for idx, row in final_selection.iterrows():
    print(f"  {row['gene_symbol']:12s} | {row['manual_class']:15s} | Skor: {row['alzheimer_score']:.3f} | {row['gene_name']}")

# UniProt ID kontrolü
missing_uniprot = final_selection[final_selection["uniprot_id"].isna()]
if len(missing_uniprot) > 0:
    print(f"\n⚠️ UYARI: {len(missing_uniprot)} gende UniProt ID eksik:")
    print(missing_uniprot[["gene_symbol", "manual_class"]])

    # Eksik olanları çıkar
    final_selection = final_selection[final_selection["uniprot_id"].notna()].copy()
    print(f"\n✓ UniProt ID eksik olanlar çıkarıldı. Kalan: {len(final_selection)} gen")

# ============================================
# DOSYALARI KAYDET
# ============================================
print("\n" + "=" * 80)
print("💾 DOSYALARI KAYDEDİYORUZ...")
print("=" * 80)

# Ana dosya
final_selection.to_csv("strategy_b_top15_targets.csv", index=False)
print(f"✓ Ana dosya: strategy_b_top15_targets.csv ({len(final_selection)} gen)")

# UniProt ID listesi
uniprot_ids = final_selection["uniprot_id"].unique().tolist()
with open("strategy_b_uniprot_ids.txt", "w") as f:
    f.write("\n".join(uniprot_ids))
print(f"✓ UniProt listesi: strategy_b_uniprot_ids.txt ({len(uniprot_ids)} ID)")

# Gen sembolleri listesi
gene_symbols = final_selection["gene_symbol"].unique().tolist()
with open("strategy_b_gene_symbols.txt", "w") as f:
    f.write("\n".join(gene_symbols))
print(f"✓ Gen sembolleri: strategy_b_gene_symbols.txt ({len(gene_symbols)} gen)")

# Özet rapor
with open("strategy_b_selection_report.txt", "w") as f:
    f.write("ALZHEIMER İLAÇ KEŞFİ - STRATEJİ B HEDEF SEÇİMİ\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Toplam Hedef: {len(final_selection)} gen\n\n")
    f.write("Sınıf Dağılımı:\n")
    for class_name, count in final_selection["manual_class"].value_counts().items():
        f.write(f"  {class_name}: {count} gen\n")
    f.write("\n\nSeçilen Genler:\n")
    f.write("-" * 60 + "\n")
    for _, row in final_selection.iterrows():
        f.write(f"{row['gene_symbol']:12s} | {row['manual_class']:15s} | "
                f"Skor: {row['alzheimer_score']:.3f} | UniProt: {row['uniprot_id']}\n")

print("✓ Özet rapor: strategy_b_selection_report.txt")

print(f"""
 ÖZET:
- Toplam hedef: {len(final_selection)} gen
- UniProt ID'si olan: {len(uniprot_ids)} protein

OLUŞTURULAN DOSYALAR:
1. strategy_b_top15_targets.csv - Detaylı gen bilgileri
2. strategy_b_uniprot_ids.txt - ChEMBL sorgusu için
3. strategy_b_gene_symbols.txt - Gen sembolleri
4. strategy_b_selection_report.txt - Özet rapor

""")



In [ ]:
!pip install chembl_webresource_client


In [ ]:
"""
STRATEJİ B - ADIM 2: ChEMBL'DEN VERİ ÇEKME
==========================================
15 Alzheimer hedefi için ilaç aktivite verilerini topluyoruz
"""

import pandas as pd
from chembl_webresource_client.new_client import new_client
import time

print("=" * 80)
print("ADIM 2: ChEMBL'DEN VERİ ÇEKME")
print("=" * 80)

# ============================================
# HAZIRLIK: Dosyaları Yükle
# ============================================
print("\n Dosyalar yükleniyor...")

try:
    # Gen bilgilerini yükle
    targets_df = pd.read_csv("strategy_b_top15_targets.csv")
    print(f"✓ {len(targets_df)} hedef gen yüklendi")

    # UniProt listesini yükle
    with open("strategy_b_uniprot_ids.txt", "r") as f:
        uniprot_list = [line.strip() for line in f.readlines()]
    print(f"✓ {len(uniprot_list)} UniProt ID yüklendi")

except FileNotFoundError as e:
    print(f" HATA: Dosya bulunamadı - {e}")
    print("Lütfen önce ADIM 1'i çalıştırın!")
    exit()

# ============================================
# ChEMBL İstemcilerini Hazırla
# ============================================
print("\n ChEMBL bağlantısı kuruluyor...")

target_client = new_client.target
activity_client = new_client.activity

print("✓ ChEMBL istemcileri hazır")

# ============================================
# VERİ ÇEKME DÖNGÜSÜ
# ============================================
print("\n" + "=" * 80)
print(" VERİ ÇEKME BAŞLIYOR...")
print("=" * 80)

all_activities = []
target_stats = []

for idx, row in targets_df.iterrows():
    gene_symbol = row['gene_symbol']
    uniprot_id = row['uniprot_id']
    manual_class = row['manual_class']

    print(f"\n[{idx+1}/{len(targets_df)}]  {gene_symbol} ({manual_class})")
    print(f"    UniProt ID: {uniprot_id}")

    # ----------------------------------------
    # 1. ChEMBL Hedefini Bul
    # ----------------------------------------
    try:
        target_query = target_client.filter(
            target_components__accession=uniprot_id
        ).only(['target_chembl_id', 'pref_name', 'organism', 'target_type'])

        targets = list(target_query)

        if not targets:
            print(f"     ChEMBL'de hedef bulunamadı")
            target_stats.append({
                'gene_symbol': gene_symbol,
                'uniprot_id': uniprot_id,
                'manual_class': manual_class,
                'chembl_id': None,
                'chembl_name': None,
                'activity_count': 0,
                'status': 'Not Found'
            })
            continue

        # İlk hedefi al (genellikle en doğru olanı)
        target_chembl_id = targets[0]['target_chembl_id']
        target_name = targets[0]['pref_name']
        target_type = targets[0].get('target_type', 'Unknown')

        print(f"    ✓ ChEMBL ID: {target_chembl_id}")
        print(f"    ✓ Hedef adı: {target_name}")
        print(f"    ✓ Hedef tipi: {target_type}")

    except Exception as e:
        print(f"     Hata (hedef arama): {e}")
        target_stats.append({
            'gene_symbol': gene_symbol,
            'uniprot_id': uniprot_id,
            'manual_class': manual_class,
            'chembl_id': None,
            'chembl_name': None,
            'activity_count': 0,
            'status': f'Error: {str(e)[:50]}'
        })
        continue

    # ----------------------------------------
    # 2. Aktivite Verilerini Çek
    # ----------------------------------------
    try:
        print(f"     Aktivite verileri çekiliyor...")

        # IC50 verileri (en yaygın)
        activities = activity_client.filter(
            target_chembl_id=target_chembl_id,
            standard_type="IC50",
            standard_relation="=",  # Sadece kesin değerler
            pchembl_value__isnull=False,  # pChEMBL değeri olmalı
            standard_units="nM"  # Nanomolar birim
        ).only([
            'molecule_chembl_id',
            'canonical_smiles',
            'standard_value',
            'standard_units',
            'pchembl_value',
            'activity_comment',
            'assay_chembl_id',
            'assay_description',
            'assay_type'
        ])

        activity_list = list(activities)

        if not activity_list:
            print(f"     IC50 verisi bulunamadı")
            target_stats.append({
                'gene_symbol': gene_symbol,
                'uniprot_id': uniprot_id,
                'manual_class': manual_class,
                'chembl_id': target_chembl_id,
                'chembl_name': target_name,
                'activity_count': 0,
                'status': 'No IC50 Data'
            })
            continue

        # DataFrame'e çevir
        df_temp = pd.DataFrame(activity_list)
        import re
        import numpy as np
        def clean_standard_value(value):
            if isinstance(value, str):
                parts = re.findall(r'\d+\.?\d*', value)
                if parts:
                     return float(parts[0])  # ilk sayıyı al
                else:
                    return np.nan
            return value

        df_temp["standard_value"] = df_temp["standard_value"].apply(clean_standard_value)
        df_temp["standard_value"] = pd.to_numeric(df_temp["standard_value"], errors="coerce")


        # Hedef bilgilerini ekle
        df_temp['target_chembl_id'] = target_chembl_id
        df_temp['target_name'] = target_name
        df_temp['target_type'] = target_type
        df_temp['uniprot_id'] = uniprot_id
        df_temp['gene_symbol'] = gene_symbol
        df_temp['manual_class'] = manual_class

        all_activities.append(df_temp)

        print(f"     {len(df_temp)} aktivite verisi bulundu")
        print(f"     İstatistikler:")
        print(f"       - Ortalama IC50: {df_temp['standard_value'].mean():.2f} nM")
        print(f"       - Medyan pChEMBL: {df_temp['pchembl_value'].median():.2f}")

        target_stats.append({
            'gene_symbol': gene_symbol,
            'uniprot_id': uniprot_id,
            'manual_class': manual_class,
            'chembl_id': target_chembl_id,
            'chembl_name': target_name,
            'activity_count': len(df_temp),
            'status': 'Success'
        })

    except Exception as e:
        print(f"     Hata (aktivite çekme): {e}")
        target_stats.append({
            'gene_symbol': gene_symbol,
            'uniprot_id': uniprot_id,
            'manual_class': manual_class,
            'chembl_id': target_chembl_id,
            'chembl_name': target_name,
            'activity_count': 0,
            'status': f'Error: {str(e)[:50]}'
        })

    # API'yi yormamak için kısa bekleme
    time.sleep(0.5)

# ============================================
# VERİLERİ BİRLEŞTİR VE KAYDET
# ============================================
print("\n\n" + "=" * 80)
print(" VERİLER BİRLEŞTİRİLİYOR VE KAYDEDİLİYOR...")
print("=" * 80)

if all_activities:
    # Tüm aktiviteleri birleştir
    df_all_activities = pd.concat(all_activities, ignore_index=True)

    print(f"\n TOPLAM {len(df_all_activities)} aktivite verisi toplandı!")
    print(f"   - Tekil molekül sayısı: {df_all_activities['molecule_chembl_id'].nunique()}")
    print(f"   - Tekil hedef sayısı: {df_all_activities['target_chembl_id'].nunique()}")

    # Ham veriyi kaydet
    df_all_activities.to_csv("strategy_b_chembl_raw_data.csv", index=False)
    print("\n✓ Ham veri kaydedildi: strategy_b_chembl_raw_data.csv")

    # ----------------------------------------
    # Hedef Bazlı İstatistikler
    # ----------------------------------------
    print("\n HEDEF BAZLI İSTATİSTİKLER:")
    print("-" * 80)
                             # Sayısal dönüşüm
    df_all_activities["standard_value"] = (
     df_all_activities["standard_value"]
     .astype(str)
     .str.replace(",", ".", regex=False)
 )
    df_all_activities["standard_value"] = pd.to_numeric(df_all_activities["standard_value"], errors="coerce")
    df_all_activities["pchembl_value"] = pd.to_numeric(df_all_activities["pchembl_value"], errors="coerce")

    target_summary = df_all_activities.groupby(['gene_symbol', 'manual_class']).agg({
        'molecule_chembl_id': 'count',
        'standard_value': ['mean', 'median', 'std'],
        'pchembl_value': ['mean', 'median']
    }).round(2)

    target_summary.columns = ['Count', 'IC50_Mean', 'IC50_Median', 'IC50_Std',
                               'pChEMBL_Mean', 'pChEMBL_Median']
    target_summary = target_summary.reset_index()
    target_summary = target_summary.sort_values('Count', ascending=False)

    print(target_summary.to_string(index=False))

    target_summary.to_csv("strategy_b_target_statistics.csv", index=False)
    print("\n✓ Hedef istatistikleri: strategy_b_target_statistics.csv")

else:
    print("\n HİÇ AKTİVİTE VERİSİ BULUNAMADI!")
    print("Olası nedenler:")
    print("  - UniProt ID'leri yanlış olabilir")
    print("  - ChEMBL'de bu hedefler için IC50 verisi yok")
    print("  - İnternet bağlantısı sorunu")


In [ ]:
import pandas as pd

df3=pd.read_csv("strategy_b_chembl_raw_data.csv")
df3

In [ ]:
"""
STRATEJİ B - ADIM 3: VERİ TEMİZLEME VE ÖN İŞLEME
================================================
40,699 aktivite verisini model için hazırlıyoruz
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 80)
print("🧹 ADIM 3: VERİ TEMİZLEME VE ÖN İŞLEME")
print("=" * 80)

# ============================================
# 1. HAM VERİYİ YÜKLE
# ============================================
print("\nHam veri yükleniyor...")

try:
    df = pd.read_csv("strategy_b_chembl_raw_data.csv")
    print(f"✓ {len(df)} kayıt yüklendi")
    print(f"✓ {df.shape[1]} sütun var")
except FileNotFoundError:
    print(" HATA: 'strategy_b_chembl_raw_data.csv' bulunamadı!")
    print("Lütfen önce ADIM 2'yi çalıştırın!")
    exit()

print(f"\n İlk bakış:")
print(df.head(3))

print(f"\n Sütunlar:")
print(df.columns.tolist())

# ============================================
# 2. EKSİK VERİ ANALİZİ
# ============================================
print("\n" + "=" * 80)
print(" EKSİK VERİ ANALİZİ")
print("=" * 80)

missing_stats = pd.DataFrame({
    'Sütun': df.columns,
    'Eksik': df.isnull().sum(),
    'Eksik %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_stats = missing_stats[missing_stats['Eksik'] > 0].sort_values('Eksik', ascending=False)

if len(missing_stats) > 0:
    print("\nEksik veriler:")
    print(missing_stats.to_string(index=False))
else:
    print("\n Hiç eksik veri yok!")

# ============================================
# 3. KRİTİK SÜTUNLARI KONTROL ET
# ============================================
print("\n" + "=" * 80)
print(" KRİTİK SÜTUN KONTROLÜ")
print("=" * 80)

critical_columns = ['molecule_chembl_id', 'canonical_smiles', 'standard_value',
                    'pchembl_value', 'target_chembl_id', 'gene_symbol']

print("\nKritik sütunlar için eksik veri:")
for col in critical_columns:
    if col in df.columns:
        missing = df[col].isnull().sum()
        print(f"  {col:25s}: {missing:6d} eksik ({missing/len(df)*100:.1f}%)")

# ============================================
# 4. VERİ TEMİZLEME - AŞAMA 1
# ============================================
print("\n" + "=" * 80)
print("🧹 VERİ TEMİZLEME - AŞAMA 1: Temel Filtreleme")
print("=" * 80)

df_clean = df.copy()
initial_count = len(df_clean)

# 1. SMILES eksik olanları çıkar
df_clean = df_clean[df_clean['canonical_smiles'].notna()].copy()
print(f"1. SMILES eksik olanlar çıkarıldı: {initial_count - len(df_clean)} kayıt")

# 2. Standard value eksik olanları çıkar
df_clean = df_clean[df_clean['standard_value'].notna()].copy()
print(f"2. IC50 değeri eksik olanlar çıkarıldı: {initial_count - len(df_clean)} kayıt (kalan: {len(df_clean)})")

# 3. Standard value'yu numeric'e çevir
df_clean['standard_value'] = pd.to_numeric(df_clean['standard_value'], errors='coerce')
df_clean = df_clean[df_clean['standard_value'].notna()].copy()
print(f"3. Numeric olmayan IC50 değerleri çıkarıldı: (kalan: {len(df_clean)})")

# 4. pChEMBL value'yu numeric'e çevir
if 'pchembl_value' in df_clean.columns:
    df_clean['pchembl_value'] = pd.to_numeric(df_clean['pchembl_value'], errors='coerce')

print(f"\n Temizleme sonrası: {len(df_clean)} kayıt ({len(df_clean)/initial_count*100:.1f}%)")

# ============================================
# 5. IC50 DEĞER DAĞILIMI ANALİZİ
# ============================================
print("\n" + "=" * 80)
print("📊 IC50 DEĞER DAĞILIMI")
print("=" * 80)

ic50_stats = df_clean['standard_value'].describe()
print("\nIC50 İstatistikleri (nM):")
print(ic50_stats)

# Aşırı uç değerleri tespit et
q1 = df_clean['standard_value'].quantile(0.25)
q3 = df_clean['standard_value'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 3 * iqr  # 3*IQR (daha toleranslı)
upper_bound = q3 + 3 * iqr

outliers = df_clean[(df_clean['standard_value'] < lower_bound) |
                    (df_clean['standard_value'] > upper_bound)]

print(f"\n Aşırı uç değerler:")
print(f"  Alt sınır: {lower_bound:.2f} nM")
print(f"  Üst sınır: {upper_bound:.2f} nM")
print(f"  Aşırı uç kayıt sayısı: {len(outliers)} ({len(outliers)/len(df_clean)*100:.1f}%)")

# ============================================
# 6. VERİ TEMİZLEME - AŞAMA 2
# ============================================
print("\n" + "=" * 80)
print(" VERİ TEMİZLEME - AŞAMA 2: Mantıklı Değer Aralığı")
print("=" * 80)

# IC50 için mantıklı aralık: 0.01 nM - 100,000 nM (100 µM)
df_clean = df_clean[
    (df_clean['standard_value'] >= 0.01) &
    (df_clean['standard_value'] <= 100000)
].copy()

print(f"IC50 aralığı filtresi (0.01 - 100,000 nM): {len(df_clean)} kayıt kaldı")

# ============================================
# 7. TEKRARLI VERİLERİ TEMİZLE
# ============================================
print("\n" + "=" * 80)
print("VERİ TEMİZLEME - AŞAMA 3: Tekrarları Temizleme")
print("=" * 80)

# Aynı molekül-hedef çiftinin tekrarlarını kontrol et
duplicates = df_clean.duplicated(subset=['molecule_chembl_id', 'target_chembl_id'], keep=False)
print(f"\nTekrarlı kayıt sayısı: {duplicates.sum()}")

# Tekrarları birleştir (medyan değer al)
print("\nTekrarlar birleştiriliyor (medyan değer)...")
df_dedup = df_clean.groupby(['molecule_chembl_id', 'target_chembl_id']).agg({
    'canonical_smiles': 'first',
    'standard_value': 'median',
    'pchembl_value': 'median' if 'pchembl_value' in df_clean.columns else 'first',
    'standard_units': 'first',
    'target_name': 'first',
    'target_type': 'first',
    'uniprot_id': 'first',
    'gene_symbol': 'first',
    'manual_class': 'first'
}).reset_index()

print(f" Tekilleştirme sonrası: {len(df_dedup)} kayıt")
print(f"   Önceki: {len(df_clean)} → Sonraki: {len(df_dedup)} (Fark: {len(df_clean) - len(df_dedup)})")

# ============================================
# 8. BİOACTİVİTY SINIFLANDIRMASI
# ============================================
print("\n" + "=" * 80)
print("🏷️ BİOACTİVİTY SINIFLANDIRMASI")
print("=" * 80)

def classify_bioactivity(ic50_value):
    """
    IC50 değerine göre sınıflandırma:
    - Active: IC50 <= 1000 nM (1 µM)
    - Intermediate: 1000 < IC50 < 10000 nM
    - Inactive: IC50 >= 10000 nM (10 µM)
    """
    if ic50_value <= 1000:
        return "active"
    elif ic50_value >= 10000:
        return "inactive"
    else:
        return "intermediate"

df_dedup['bioactivity_class'] = df_dedup['standard_value'].apply(classify_bioactivity)

print("\nSınıf dağılımı:")
class_counts = df_dedup['bioactivity_class'].value_counts()
for cls, count in class_counts.items():
    print(f"  {cls:15s}: {count:6d} ({count/len(df_dedup)*100:.1f}%)")

# ============================================
# 9. 2-SINIF VERSİYONU (Active vs Inactive)
# ============================================
print("\n" + "=" * 80)
print(" 2-SINIF VERSİYONU (Active vs Inactive)")
print("=" * 80)

df_2class = df_dedup[df_dedup['bioactivity_class'] != 'intermediate'].copy()

print(f"\nIntermediate veriler çıkarıldı: {len(df_2class)} kayıt kaldı")
print("\n2-Sınıf dağılımı:")
class_counts_2 = df_2class['bioactivity_class'].value_counts()
for cls, count in class_counts_2.items():
    print(f"  {cls:15s}: {count:6d} ({count/len(df_2class)*100:.1f}%)")

# Dengesizlik kontrolü
ratio = class_counts_2.max() / class_counts_2.min()
print(f"\nSınıf dengesizlik oranı: {ratio:.2f}:1")
if ratio > 3:
    print(" UYARI: Sınıflar dengesiz! Model eğitiminde SMOTE veya class_weight kullanın")
else:
    print(" Sınıflar makul derecede dengeli")

# ============================================
# 10. HEDEF BAZINDA ANALİZ
# ============================================
print("\n" + "=" * 80)
print("HEDEF BAZINDA VERİ DAĞILIMI")
print("=" * 80)

target_distribution = df_2class.groupby(['gene_symbol', 'manual_class']).agg({
    'molecule_chembl_id': 'count',
    'standard_value': ['mean', 'median'],
    'bioactivity_class': lambda x: (x == 'active').sum()
}).round(2)

target_distribution.columns = ['Total_Count', 'IC50_Mean', 'IC50_Median', 'Active_Count']
target_distribution['Active_%'] = (target_distribution['Active_Count'] /
                                    target_distribution['Total_Count'] * 100).round(1)
target_distribution = target_distribution.reset_index()
target_distribution = target_distribution.sort_values('Total_Count', ascending=False)

print("\n")
print(target_distribution.to_string(index=False))

# Yetersiz veri olan hedefleri kontrol et
low_data_targets = target_distribution[target_distribution['Total_Count'] < 50]
if len(low_data_targets) > 0:
    print(f"\n Uyarı: {len(low_data_targets)} hedefte <50 veri var:")
    print(low_data_targets[['gene_symbol', 'Total_Count']].to_string(index=False))

# ============================================
# 11. DOSYALARI KAYDET
# ============================================
print("\n" + "=" * 80)
print(" TEMİZ VERİLER KAYDEDİLİYOR...")
print("=" * 80)

# 1. Tam temiz veri (3 sınıf)
df_dedup.to_csv("strategy_b_data_clean_3class.csv", index=False)
print(f"✓ 3-sınıf veri: strategy_b_data_clean_3class.csv ({len(df_dedup)} kayıt)")

# 2. 2-sınıf veri (active/inactive)
df_2class.to_csv("strategy_b_data_clean_2class.csv", index=False)
print(f"✓ 2-sınıf veri: strategy_b_data_clean_2class.csv ({len(df_2class)} kayıt)")

# 3. Hedef dağılımı
target_distribution.to_csv("strategy_b_target_distribution.csv", index=False)
print("✓ Hedef dağılımı: strategy_b_target_distribution.csv")

# 4. Özet istatistikler
summary_stats = pd.DataFrame({
    'Metric': [
        'Ham Veri',
        'Temizlik Sonrası',
        '2-Sınıf Veri',
        'Tekil Molekül',
        'Tekil Hedef',
        'Active Oran (%)',
        'Inactive Oran (%)'
    ],
    'Value': [
        len(df),
        len(df_dedup),
        len(df_2class),
        df_2class['molecule_chembl_id'].nunique(),
        df_2class['gene_symbol'].nunique(),
        f"{(df_2class['bioactivity_class'] == 'active').sum() / len(df_2class) * 100:.1f}",
        f"{(df_2class['bioactivity_class'] == 'inactive').sum() / len(df_2class) * 100:.1f}"
    ]
})

summary_stats.to_csv("strategy_b_cleaning_summary.csv", index=False)
print("✓ Temizleme özeti: strategy_b_cleaning_summary.csv")

# ============================================
# 12. GÖRSELLEŞTİRME
# ============================================
print("\n" + "=" * 80)
print(" GÖRSELLEŞTİRMELER OLUŞTURULUYOR...")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. IC50 dağılımı (log scale)
axes[0, 0].hist(np.log10(df_2class['standard_value']), bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('log10(IC50 nM)', fontsize=12)
axes[0, 0].set_ylabel('Frekans', fontsize=12)
axes[0, 0].set_title('IC50 Değer Dağılımı (Log Scale)', fontsize=14, fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# 2. Bioactivity sınıf dağılımı
class_counts_2.plot(kind='bar', ax=axes[0, 1], color=['green', 'red'], edgecolor='black')
axes[0, 1].set_xlabel('Bioactivity Sınıfı', fontsize=12)
axes[0, 1].set_ylabel('Kayıt Sayısı', fontsize=12)
axes[0, 1].set_title('Bioactivity Sınıf Dağılımı', fontsize=14, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Hedef bazında veri sayısı
top_targets = target_distribution.nlargest(15, 'Total_Count')
axes[1, 0].barh(top_targets['gene_symbol'], top_targets['Total_Count'], color='steelblue', edgecolor='black')
axes[1, 0].set_xlabel('Veri Sayısı', fontsize=12)
axes[1, 0].set_ylabel('Gen', fontsize=12)
axes[1, 0].set_title('Hedef Bazında Veri Dağılımı (Top 15)', fontsize=14, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Active oran hedeflere göre
axes[1, 1].barh(top_targets['gene_symbol'], top_targets['Active_%'], color='lightgreen', edgecolor='black')
axes[1, 1].set_xlabel('Active Oran (%)', fontsize=12)
axes[1, 1].set_ylabel('Gen', fontsize=12)
axes[1, 1].set_title('Hedef Bazında Active Oranı (Top 15)', fontsize=14, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_b_data_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Görselleştirme: strategy_b_data_analysis.png")
plt.close()


print(f"""
 VERİ TEMİZLEME ÖZETİ:
{'='*50}
Ham Veri:              {len(df):,} kayıt
Temizlik Sonrası:      {len(df_dedup):,} kayıt
2-Sınıf Veri:          {len(df_2class):,} kayıt
Veri Kaybı:            {(1 - len(df_2class)/len(df))*100:.1f}%

Tekil Molekül:         {df_2class['molecule_chembl_id'].nunique():,}
Tekil Hedef:           {df_2class['gene_symbol'].nunique()}

Active:                {(df_2class['bioactivity_class'] == 'active').sum():,} ({(df_2class['bioactivity_class'] == 'active').sum()/len(df_2class)*100:.1f}%)
Inactive:              {(df_2class['bioactivity_class'] == 'inactive').sum():,} ({(df_2class['bioactivity_class'] == 'inactive').sum()/len(df_2class)*100:.1f}%)

 OLUŞTURULAN DOSYALAR:
{'='*50}
1. strategy_b_data_clean_3class.csv - 3 sınıflı veri
2. strategy_b_data_clean_2class.csv - 2 sınıflı veri (MODELİ BU İLE EĞİTİN)
3. strategy_b_target_distribution.csv - Hedef dağılım tablosu
4. strategy_b_cleaning_summary.csv - Temizleme özeti
5. strategy_b_data_analysis.png - Görselleştirme


""")




In [ ]:
pip install rdkit

In [ ]:
"""
STRATEJİ B - GNN YAKLAŞIMI: GRAPH NEURAL NETWORK
=================================================
PyTorch Geometric ile moleküler graf öğrenimi
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, BatchNorm
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("🧠 GNN İLE MOLEKÜLER BİYOAKTİVİTE TAHMİNİ")
print("=" * 80)

# GPU kontrolü
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan cihaz: {device}")

# ============================================
# 1. MOLEKÜLDEN GRAF OLUŞTURMA
# ============================================

def mol_to_graph(smiles):
    """SMILES'tan PyTorch Geometric Data nesnesi oluşturur"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Atom özellikleri (node features)
        atom_features = []
        for atom in mol.GetAtoms():
            features = [
                atom.GetAtomicNum(),  # Atom numarası
                atom.GetDegree(),  # Bağ sayısı
                atom.GetFormalCharge(),  # Yük
                atom.GetHybridization().real,  # Hibridizasyon
                atom.GetIsAromatic() * 1,  # Aromatik mi?
                atom.GetTotalNumHs(),  # H sayısı
                atom.IsInRing() * 1,  # Halka içinde mi?
            ]
            atom_features.append(features)

        x = torch.tensor(atom_features, dtype=torch.float)

        # Bağ bilgileri (edge indices)
        edge_index = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            edge_index.append([i, j])
            edge_index.append([j, i])  # Yönsüz graf için

        if len(edge_index) == 0:
            return None

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        return Data(x=x, edge_index=edge_index)

    except Exception as e:
        return None


# ============================================
# 2. GNN MODELİ
# ============================================

class MolecularGNN(nn.Module):
    def __init__(self, num_node_features, hidden_channels=128, num_classes=2):
        super(MolecularGNN, self).__init__()

        # GCN katmanları
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)

        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.bn3 = BatchNorm(hidden_channels)

        # Fully connected katmanlar
        self.fc1 = nn.Linear(hidden_channels, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # GCN katmanları
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        # Global pooling (graf seviyesi temsil)
        x = global_mean_pool(x, batch)

        # Sınıflandırma
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        return x


# ============================================
# 3. VERİ YÜKLEME VE HAZIRLIK
# ============================================

print("\n Veri yükleniyor...")
df = pd.read_csv("strategy_b_data_clean_2class.csv")
print(f"✓ {len(df)} kayıt yüklendi")

# Sınıf dağılımı
print(f"\nSınıf dağılımı:")
print(df['bioactivity_class'].value_counts())

# ============================================
# 4. GRAF VERİSİ OLUŞTURMA
# ============================================

print("\n Moleküller graf formatına dönüştürülüyor...")
graph_data_list = []
labels = []
failed_count = 0

for idx, row in df.iterrows():
    graph = mol_to_graph(row['canonical_smiles'])
    if graph is not None:
        label = 1 if row['bioactivity_class'] == 'active' else 0
        graph.y = torch.tensor([label], dtype=torch.long)
        graph_data_list.append(graph)
        labels.append(label)
    else:
        failed_count += 1

    if (idx + 1) % 2000 == 0:
        print(f"  İşlenen: {idx + 1:,}/{len(df):,} - Başarılı: {len(graph_data_list):,}")

print(f"\n Graf oluşturma tamamlandı!")
print(f"   Başarılı: {len(graph_data_list):,}")
print(f"   Başarısız: {failed_count:,}")

# ============================================
# 5. TRAIN-TEST SPLIT
# ============================================

train_graphs, test_graphs, train_labels, test_labels = train_test_split(
    graph_data_list, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\n✓ Train set: {len(train_graphs):,}")
print(f"✓ Test set: {len(test_graphs):,}")

# DataLoader'lar oluştur
train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=64, shuffle=False)

# ============================================
# 6. MODEL EĞİTİMİ
# ============================================

print("\n" + "=" * 80)
print("GNN MODEL EĞİTİMİ")
print("=" * 80)

# Model oluştur
num_node_features = graph_data_list[0].x.shape[1]
model = MolecularGNN(num_node_features=num_node_features, hidden_channels=128).to(device)

# Loss ve optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                        factor=0.5, patience=10)

# Eğitim fonksiyonu
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        correct += (pred == data.y).sum().item()
        total += data.num_graphs

    return total_loss / total, correct / total

# Test fonksiyonu
def test_epoch(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)
            pred = out.argmax(dim=1)
            probs = F.softmax(out, dim=1)[:, 1]

            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return all_preds, all_labels, all_probs

# Eğitim döngüsü
print("\nModel eğitiliyor...")
num_epochs = 100
best_test_auc = 0
train_losses = []
train_accs = []
test_aucs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    test_preds, test_labels_epoch, test_probs = test_epoch(model, test_loader)

    test_auc = roc_auc_score(test_labels_epoch, test_probs)
    test_acc = accuracy_score(test_labels_epoch, test_preds)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_aucs.append(test_auc)

    scheduler.step(train_loss)

    if test_auc > best_test_auc:
        best_test_auc = test_auc
        torch.save(model.state_dict(), 'strategy_b_gnn_best.pt')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | Test AUC: {test_auc:.4f}")

print(f"\n Eğitim tamamlandı! En iyi Test AUC: {best_test_auc:.4f}")

# ============================================
# 7. EN İYİ MODELİ YÜKLE VE DEĞERLENDİR
# ============================================

print("\n" + "=" * 80)
print(" MODEL DEĞERLENDİRME")
print("=" * 80)

model.load_state_dict(torch.load('strategy_b_gnn_best.pt'))
test_preds, test_labels_final, test_probs = test_epoch(model, test_loader)

# Metrikler
test_acc = accuracy_score(test_labels_final, test_preds)
test_prec = precision_score(test_labels_final, test_preds)
test_rec = recall_score(test_labels_final, test_preds)
test_auc = roc_auc_score(test_labels_final, test_probs)

print(f"\n PERFORMANS METRİKLERİ:")
print("=" * 50)
print(f"Test Accuracy:   {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"Test Precision:  {test_prec:.4f}")
print(f"Test Recall:     {test_rec:.4f}")
print(f"Test ROC-AUC:    {test_auc:.4f}")

print("\n📋 Classification Report:")
print(classification_report(test_labels_final, test_preds,
                          target_names=['Inactive', 'Active']))

cm = confusion_matrix(test_labels_final, test_preds)
print("\n Confusion Matrix:")
print(f"                Predicted")
print(f"              Inactive  Active")
print(f"Actual Inactive {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"       Active   {cm[1,0]:5d}   {cm[1,1]:5d}")

# ============================================
# 8. GÖRSELLEŞTİRME
# ============================================

print("\n Görselleştirmeler oluşturuluyor...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Training curves
axes[0, 0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Accuracy curves
axes[0, 1].plot(train_accs, label='Train Acc', linewidth=2)
axes[0, 1].plot(test_aucs, label='Test AUC', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('Training Progress')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Inactive', 'Active'],
            yticklabels=['Inactive', 'Active'])
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_title('Confusion Matrix')

# 4. Probability distribution
axes[1, 1].hist([p for p, l in zip(test_probs, test_labels_final) if l == 0],
                bins=30, alpha=0.6, label='Inactive', color='red', edgecolor='black')
axes[1, 1].hist([p for p, l in zip(test_probs, test_labels_final) if l == 1],
                bins=30, alpha=0.6, label='Active', color='green', edgecolor='black')
axes[1, 1].set_xlabel('Predicted Probability (Active)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Prediction Distribution')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('strategy_b_gnn_results.png', dpi=300, bbox_inches='tight')
print("✓ Görsel kaydedildi: strategy_b_gnn_results.png")

# ============================================
# 9. SONUÇLARI KAYDET
# ============================================

print("\n Sonuçlar kaydediliyor...")

metrics = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test Precision', 'Test Recall', 'Test ROC-AUC'],
    'Value': [test_acc, test_prec, test_rec, test_auc]
})
metrics.to_csv('strategy_b_gnn_metrics.csv', index=False)
print("✓ Metrikler kaydedildi: strategy_b_gnn_metrics.csv")

print("\n" + "=" * 80)
print(" GNN MODEL EĞİTİMİ TAMAMLANDI!")
print("=" * 80)
print(f"""
 ÖZET:
Toplam Veri:        {len(graph_data_list):,} molekül
Train Set:          {len(train_graphs):,}
Test Set:           {len(test_graphs):,}

Model:              Graph Convolutional Network (3 layers)
Hidden Channels:    128
Epochs:             {num_epochs}

 PERFORMANS:
Test Accuracy:      {test_acc:.3f} ({test_acc*100:.1f}%)
Test Precision:     {test_prec:.3f}
Test Recall:        {test_rec:.3f}
Test ROC-AUC:       {test_auc:.3f}
""")

In [ ]:


import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("ALZHEİMER İLAÇ KEŞFİ - TAM PİPELİNE (4 MODEL)")
print("XGBoost | LightGBM | Random Forest | Logistic Regression")
print("Multi-target aware | Scaffold split | Macro F1 threshold")
print("=" * 80)

# ─────────────────────────────────────────────
# 1. VERİYİ YÜKLE
# ─────────────────────────────────────────────
print("\n Veri yükleniyor...")
df = pd.read_csv("strategy_b_data_clean_2class.csv")
print(f"✓ {len(df)} kayıt yüklendi")
print(f"  Active: {(df['bioactivity_class']=='active').sum():,} | "
      f"Inactive: {(df['bioactivity_class']=='inactive').sum():,}")

# ─────────────────────────────────────────────
# 2. VERİ TEMİZLEME
# ─────────────────────────────────────────────
print("\n Çelişen SMILES temizleniyor...")

conflict_check = df.groupby('canonical_smiles')['bioactivity_class'].nunique()
conflict_smiles = conflict_check[conflict_check > 1].index
df_clean = df[~df['canonical_smiles'].isin(conflict_smiles)].copy()

print(f"  Çelişen SMILES: {len(conflict_smiles)} adet çıkarıldı")
print(f"  Temiz kayıt: {len(df_clean):,}")

# ─────────────────────────────────────────────
# 3. ÖZELLİK ÇIKARIMI
# ─────────────────────────────────────────────
print("\n Özellikler hesaplanıyor...")
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

# Target one-hot encoding
target_dummies = pd.get_dummies(df_clean['target_chembl_id'], prefix='target')
target_cols = target_dummies.columns.tolist()
df_clean = pd.concat([df_clean.reset_index(drop=True),
                      target_dummies.reset_index(drop=True)], axis=1)

def get_scaffold(smiles):
    """Murcko scaffold çıkarır — scaffold split için kullanılır."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return smiles

def calculate_features(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        desc = {
            'MW':                 Descriptors.MolWt(mol),
            'LogP':               Descriptors.MolLogP(mol),
            'NumHDonors':         Descriptors.NumHDonors(mol),
            'NumHAcceptors':      Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds':  Descriptors.NumRotatableBonds(mol),
            'AromaticProportion': (len(mol.GetAromaticAtoms()) / mol.GetNumAtoms()
                                   if mol.GetNumAtoms() > 0 else 0),
            'TPSA':               Descriptors.TPSA(mol),
            'NumRings':           Descriptors.RingCount(mol),
            'MolMR':              Descriptors.MolMR(mol),
            'NumAromaticRings':   Descriptors.NumAromaticRings(mol),
            'NumAliphaticRings':  Descriptors.NumAliphaticRings(mol),
            'BertzCT':            Descriptors.BertzCT(mol),
            'HeavyAtomCount':     mol.GetNumHeavyAtoms(),
            'FractionCSP3':       Descriptors.FractionCSP3(mol),
        }
        # Morgan fingerprint (ECFP4)
        fp = morgan_gen.GetFingerprint(mol)
        for i, bit in enumerate(np.array(fp)):
            desc[f'fp_{i}'] = int(bit)
        # MACCS keys
        maccs = MACCSkeys.GenMACCSKeys(mol)
        for i, bit in enumerate(maccs):
            desc[f'maccs_{i}'] = int(bit)
        return desc
    except Exception:
        return None

features_list = []
scaffolds = []
total = len(df_clean)

for idx, row in df_clean.iterrows():
    feats = calculate_features(row['canonical_smiles'])
    if feats:
        feats['bioactivity_class'] = row['bioactivity_class']
        for tc in target_cols:
            feats[tc] = row[tc]
        features_list.append(feats)
        scaffolds.append(get_scaffold(row['canonical_smiles']))

    if (len(features_list)) % 3000 == 0 and len(features_list) > 0:
        print(f"  ✓ İşlenen: {len(features_list):,}/{total:,} "
              f"({len(features_list)/total*100:.1f}%)")

print(f"✓ Özellik çıkarımı tamamlandı: {len(features_list):,} molekül")

df_feat = pd.DataFrame(features_list)
scaffolds = np.array(scaffolds)

# ─────────────────────────────────────────────
# 4. VERİ BÖLDÜRME — Scaffold Split
# ─────────────────────────────────────────────
print("\n  Scaffold split yapılıyor (genelleme testi için)...")

feature_cols = [c for c in df_feat.columns if c != 'bioactivity_class']
X = df_feat[feature_cols].values
y = df_feat['bioactivity_class'].map({'active': 1, 'inactive': 0}).values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=scaffolds))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"  Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"  Test active: {y_test.sum():,} | Test inactive: {(y_test==0).sum():,}")

unique_train_scaffolds = len(set(scaffolds[train_idx]))
unique_test_scaffolds  = len(set(scaffolds[test_idx]))
overlap = len(set(scaffolds[train_idx]) & set(scaffolds[test_idx]))
print(f"  Train scaffolds: {unique_train_scaffolds:,} | "
      f"Test scaffolds: {unique_test_scaffolds:,} | "
      f"Örtüşme: {overlap:,} ({overlap/unique_test_scaffolds*100:.1f}%)")

# ─────────────────────────────────────────────
# 5. SMOTE DENGELEME
# ─────────────────────────────────────────────
print("\n  Veri dengeleniyor (SMOTETomek)...")

n_active   = (y_train == 1).sum()
n_inactive = (y_train == 0).sum()
target_inactive = int(n_active * 0.85)

smotetomek = SMOTETomek(
    smote=SMOTE(
        sampling_strategy={0: target_inactive},
        k_neighbors=5,
        random_state=42
    ),
    random_state=42
)
X_tr_bal, y_tr_bal = smotetomek.fit_resample(X_train, y_train)

print(f"  Dengeli train: {len(X_tr_bal):,} örnek")
print(f"  Active: {(y_tr_bal==1).sum():,} | Inactive: {(y_tr_bal==0).sum():,}")

# ─────────────────────────────────────────────
# 6. THRESHOLD OPTİMİZASYONU — Macro F1
# ─────────────────────────────────────────────
def find_best_threshold(model, X_val, y_val):
    """Macro F1 ile threshold optimize eder."""
    proba = model.predict_proba(X_val)[:, 1]
    best_thresh, best_score = 0.5, 0
    for t in np.arange(0.25, 0.75, 0.01):
        preds = (proba >= t).astype(int)
        rpt = classification_report(y_val, preds, output_dict=True, zero_division=0)
        score = rpt['macro avg']['f1-score']
        if score > best_score:
            best_score, best_thresh = score, t
    return round(best_thresh, 2), best_score

# Validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_bal, y_tr_bal,
    test_size=0.15, stratify=y_tr_bal, random_state=42
)

# ─────────────────────────────────────────────
# 6b. LOGISTIC REGRESSION İÇİN SCALER
# ─────────────────────────────────────────────
# LR mesafeye duyarlıdır; feature'ları ölçeklendirmek performansı artırır.
scaler = StandardScaler()
X_tr_scaled   = scaler.fit_transform(X_tr)
X_val_scaled  = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ─────────────────────────────────────────────
# 7. MODEL TANIMI
# ─────────────────────────────────────────────
imbalance_ratio = (y_tr_bal == 1).sum() / (y_tr_bal == 0).sum()

models = {
    'XGBoost': XGBClassifier(
        n_estimators=400,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1 / imbalance_ratio,
        eval_metric='aucpr',
        use_label_encoder=False,
        random_state=42
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=400,
        learning_rate=0.08,
        max_depth=6,
        class_weight={0: imbalance_ratio, 1: 1.0},
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        verbose=-1,
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=400,
        max_depth=None,          # tam büyüme; veri büyükse max_depth=20 dene
        min_samples_leaf=4,
        class_weight={0: imbalance_ratio, 1: 1.0},
        n_jobs=-1,
        random_state=42
    ),
    'LogisticRegression': LogisticRegression(
        C=0.1,                   # L2 regularizasyon
        class_weight={0: imbalance_ratio, 1: 1.0},
        max_iter=1000,
        solver='saga',           # büyük/seyrek veri için en hızlı solver
        n_jobs=-1,
        random_state=42
    ),
}

# ─────────────────────────────────────────────
# 8. EĞİTİM VE DEĞERLENDİRME
# ─────────────────────────────────────────────
results_summary = []
trained_models  = {}
best_thresholds = {}

SANITY_MOLECULES = {
    'Galantamin (AChE aktif)': 'COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2',
    'Donepezil (AChE aktif)':  'COc1cc2c(cc1OC)CC(CC1CCN(Cc3ccccc3)CC1)C2=O',
}

for name, model in models.items():
    print("\n" + "=" * 80)
    print(f"  {name} — EĞİTİM VE ANALİZ")
    print("=" * 80)

    # Logistic Regression ölçeklenmiş veri kullanır
    use_scaled = (name == 'LogisticRegression')
    _X_tr   = X_tr_scaled   if use_scaled else X_tr
    _X_val  = X_val_scaled  if use_scaled else X_val
    _X_test = X_test_scaled if use_scaled else X_test

    model.fit(_X_tr, y_tr)
    trained_models[name] = model

    opt_thresh, val_f1 = find_best_threshold(model, _X_val, y_val)
    best_thresholds[name] = opt_thresh
    print(f"  ✓ Optimal threshold: {opt_thresh:.2f}  (val macro F1: {val_f1:.4f})")

    y_test_proba = model.predict_proba(_X_test)[:, 1]
    y_test_pred  = (y_test_proba >= opt_thresh).astype(int)
    y_tr_pred    = model.predict(_X_tr)

    train_acc = accuracy_score(y_tr, y_tr_pred)
    test_acc  = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, zero_division=0)
    recall    = recall_score(y_test, y_test_pred, zero_division=0)
    f1        = f1_score(y_test, y_test_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_test, y_test_proba)

    results_summary.append({
        'Model':     name,
        'Threshold': opt_thresh,
        'Train Acc': train_acc,
        'Test Acc':  test_acc,
        'Precision': precision,
        'Recall':    recall,
        'F1':        f1,
        'ROC-AUC':   roc_auc
    })

    # ── Performans Metrikleri ──
    print(f"\n PERFORMANS METRİKLERİ:")
    print(f"  {'='*60}")
    print(f"  Threshold:          {opt_thresh:.2f}")
    print(f"  Train Accuracy:     {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"  Test Accuracy:      {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"  Precision:          {precision:.4f}")
    print(f"  Recall:             {recall:.4f}")
    print(f"  F1-Score:           {f1:.4f}")
    print(f"  ROC-AUC:            {roc_auc:.4f}")

    print(f"\n CLASSIFICATION REPORT:")
    report = classification_report(y_test, y_test_pred,
                                   target_names=['Inactive (0)', 'Active (1)'],
                                   digits=4)
    for line in report.split('\n'):
        print(f"  {line}")

    cm = confusion_matrix(y_test, y_test_pred)
    print(f"\n CONFUSION MATRIX:")
    print(f"  {'='*60}")
    print(f"                    Predicted")
    print(f"                    Inactive    Active")
    print(f"  Actual Inactive   {cm[0,0]:>8}    {cm[0,1]:>6}")
    print(f"  Actual Active     {cm[1,0]:>8}    {cm[1,1]:>6}")
    print(f"  {'='*60}")
    print(f"  TN={cm[0,0]:,} | FP={cm[0,1]:,} | FN={cm[1,0]:,} | TP={cm[1,1]:,}")

    # ── Sanity Check ──
    print(f"\n SANİTY CHECK (Bilinen aktif ilaçlar):")
    print(f"  {'='*60}")
    for mol_name, smiles in SANITY_MOLECULES.items():
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print(f"    {mol_name}: SMILES parse hatası")
                continue
            feats = calculate_features(smiles)
            if feats is None:
                continue
            feats.pop('bioactivity_class', None)
            for tc in target_cols:
                feats[tc] = 0
            feat_vec = np.array([[feats.get(c, 0) for c in feature_cols]])
            if use_scaled:
                feat_vec = scaler.transform(feat_vec)
            prob = model.predict_proba(feat_vec)[0, 1]
            pred = 'ACTIVE ✓' if prob >= opt_thresh else 'INACTIVE ✗'
            print(f"  {mol_name:<35} P(active)={prob:.3f} → {pred}")
        except Exception as e:
            print(f"    {mol_name}: hata — {e}")

    # ── Confusion Matrix Görseli ──
    rpt_dict    = classification_report(y_test, y_test_pred,
                                        output_dict=True, zero_division=0)
    inactive_f1 = rpt_dict['0']['f1-score']
    active_f1   = rpt_dict['1']['f1-score']

    plt.figure(figsize=(10, 8))
    ax = plt.subplot(1, 1, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Inactive (0)', 'Active (1)'],
                yticklabels=['Inactive (0)', 'Active (1)'],
                annot_kws={'size': 14, 'weight': 'bold'},
                linewidths=2, linecolor='white')
    plt.title(
        f'Confusion Matrix — {name}  [threshold={opt_thresh:.2f}]\n'
        f'Acc: {test_acc:.4f} | Prec: {precision:.4f} | '
        f'Rec: {recall:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}\n'
        f'Inactive F1: {inactive_f1:.4f} | Active F1: {active_f1:.4f}',
        fontsize=11, pad=20, weight='bold'
    )
    plt.ylabel('Gerçek Sınıf (Actual)', fontsize=12, weight='bold')
    plt.xlabel('Tahmin Edilen Sınıf (Predicted)', fontsize=12, weight='bold')
    plt.text(0.5, -0.17,
             f'TN={cm[0,0]:,} | FP={cm[0,1]:,} | FN={cm[1,0]:,} | TP={cm[1,1]:,}',
             ha='center', transform=ax.transAxes, fontsize=11, style='italic',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    plt.tight_layout()
    fname = f'confusion_matrix_{name.replace(" ", "_")}_fixed.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    print(f"\n  Grafik: {fname}")
    plt.close()

# ─────────────────────────────────────────────
# 9. KARŞILAŞTIRMA TABLOSU
# ─────────────────────────────────────────────
print("\n\n" + "=" * 80)
print(" MODEL KARŞILAŞTIRMA")
print("=" * 80)

results_df = pd.DataFrame(results_summary).sort_values('ROC-AUC', ascending=False)
print(f"\n{'Model':<22} {'Thresh':<8} {'Train Acc':<12} {'Test Acc':<12} "
      f"{'Precision':<12} {'Recall':<12} {'F1':<12} {'ROC-AUC':<10}")
print("=" * 104)
for _, row in results_df.iterrows():
    marker = " ← EN İYİ" if _ == 0 else ""
    print(f"{row['Model']:<22} {row['Threshold']:<8.2f} {row['Train Acc']:<12.4f} "
          f"{row['Test Acc']:<12.4f} {row['Precision']:<12.4f} {row['Recall']:<12.4f} "
          f"{row['F1']:<12.4f} {row['ROC-AUC']:<10.4f}{marker}")

# ─────────────────────────────────────────────
# 10. EN İYİ MODELİ PKL OLARAK KAYDET
# ─────────────────────────────────────────────
print("\n" + "=" * 80)
print(" EN İYİ MODEL KAYDEDİLİYOR")
print("=" * 80)

best_row  = results_df.iloc[0]
best_name = best_row['Model']

save_package = {
    'model':        trained_models[best_name],
    # Logistic Regression kullanılıyorsa scaler da gereklidir; diğerleri None
    'scaler':       scaler if best_name == 'LogisticRegression' else None,
    'feature_cols': feature_cols,
    'target_cols':  target_cols,
    'threshold':    best_thresholds[best_name],
    'class_map':    {1: 'active', 0: 'inactive'},
    'model_name':   best_name,
    'training_info': {
        'n_train':           len(X_tr_bal),
        'n_test':            len(X_test),
        'split_method':      'scaffold_split',
        'threshold_opt':     'macro_f1',
        'smote_ratio':       0.85,
        'conflicts_removed': len(conflict_smiles),
        'test_acc':          float(best_row['Test Acc']),
        'roc_auc':           float(best_row['ROC-AUC']),
        'f1':                float(best_row['F1']),
        'all_models_roc':    {r['Model']: round(r['ROC-AUC'], 4)
                              for _, r in results_df.iterrows()},
    }
}

pkl_path = f'best_model_{best_name.replace(" ", "_").lower()}_fixed.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(save_package, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(pkl_path) / 1e6
print(f"\n  Kaydedildi: {pkl_path}  ({size_mb:.1f} MB)")
print(f"   Model:          {best_name}")
print(f"   Threshold:      {best_thresholds[best_name]:.2f}")
print(f"   Feature sayısı: {len(feature_cols):,}")
print(f"\n  Tüm model ROC-AUC skorları:")
for _, row in results_df.iterrows():
    marker = " " if row['Model'] == best_name else ""
    print(f"    {row['Model']:<22} {row['ROC-AUC']:.4f}{marker}")
print("\n" + "=" * 80)
print("✓ Pipeline tamamlandı.")
print("=" * 80)

In [ ]:


import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("ALZHEİMER İLAÇ KEŞFİ - TAM PİPELİNE (4 MODEL)")
print("XGBoost | LightGBM | Random Forest | Logistic Regression")
print("Multi-target aware | Scaffold split | Macro F1 threshold")
print("=" * 80)

# ─────────────────────────────────────────────
# 1. VERİYİ YÜKLE
# ─────────────────────────────────────────────
print("\n Veri yükleniyor...")
df = pd.read_csv("strategy_b_data_clean_2class.csv")
print(f"✓ {len(df)} kayıt yüklendi")
print(f"  Active: {(df['bioactivity_class']=='active').sum():,} | "
      f"Inactive: {(df['bioactivity_class']=='inactive').sum():,}")

# ─────────────────────────────────────────────
# 2. VERİ TEMİZLEME
# ─────────────────────────────────────────────
print("\n Çelişen SMILES temizleniyor...")

conflict_check = df.groupby('canonical_smiles')['bioactivity_class'].nunique()
conflict_smiles = conflict_check[conflict_check > 1].index
df_clean = df[~df['canonical_smiles'].isin(conflict_smiles)].copy()

print(f"  Çelişen SMILES: {len(conflict_smiles)} adet çıkarıldı")
print(f"  Temiz kayıt: {len(df_clean):,}")

# ─────────────────────────────────────────────
# 3. ÖZELLİK ÇIKARIMI
# ─────────────────────────────────────────────
print("\n Özellikler hesaplanıyor...")
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

# Target one-hot encoding
target_dummies = pd.get_dummies(df_clean['target_chembl_id'], prefix='target')
target_cols = target_dummies.columns.tolist()
df_clean = pd.concat([df_clean.reset_index(drop=True),
                      target_dummies.reset_index(drop=True)], axis=1)

def get_scaffold(smiles):
    """Murcko scaffold çıkarır — scaffold split için kullanılır."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return smiles
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(scaffold)
    except Exception:
        return smiles

def calculate_features(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        desc = {
            'MW':                 Descriptors.MolWt(mol),
            'LogP':               Descriptors.MolLogP(mol),
            'NumHDonors':         Descriptors.NumHDonors(mol),
            'NumHAcceptors':      Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds':  Descriptors.NumRotatableBonds(mol),
            'AromaticProportion': (len(mol.GetAromaticAtoms()) / mol.GetNumAtoms()
                                   if mol.GetNumAtoms() > 0 else 0),
            'TPSA':               Descriptors.TPSA(mol),
            'NumRings':           Descriptors.RingCount(mol),
            'MolMR':              Descriptors.MolMR(mol),
            'NumAromaticRings':   Descriptors.NumAromaticRings(mol),
            'NumAliphaticRings':  Descriptors.NumAliphaticRings(mol),
            'BertzCT':            Descriptors.BertzCT(mol),
            'HeavyAtomCount':     mol.GetNumHeavyAtoms(),
            'FractionCSP3':       Descriptors.FractionCSP3(mol),
        }
        # Morgan fingerprint (ECFP4)
        fp = morgan_gen.GetFingerprint(mol)
        for i, bit in enumerate(np.array(fp)):
            desc[f'fp_{i}'] = int(bit)
        # MACCS keys
        maccs = MACCSkeys.GenMACCSKeys(mol)
        for i, bit in enumerate(maccs):
            desc[f'maccs_{i}'] = int(bit)
        return desc
    except Exception:
        return None

features_list = []
scaffolds = []
total = len(df_clean)

for idx, row in df_clean.iterrows():
    feats = calculate_features(row['canonical_smiles'])
    if feats:
        feats['bioactivity_class'] = row['bioactivity_class']
        for tc in target_cols:
            feats[tc] = row[tc]
        features_list.append(feats)
        scaffolds.append(get_scaffold(row['canonical_smiles']))

    if (len(features_list)) % 3000 == 0 and len(features_list) > 0:
        print(f"  ✓ İşlenen: {len(features_list):,}/{total:,} "
              f"({len(features_list)/total*100:.1f}%)")

print(f"✓ Özellik çıkarımı tamamlandı: {len(features_list):,} molekül")

df_feat = pd.DataFrame(features_list)
scaffolds = np.array(scaffolds)

# ─────────────────────────────────────────────
# 4. VERİ BÖLDÜRME — Scaffold Split
# ─────────────────────────────────────────────
print("\n  Scaffold split yapılıyor (genelleme testi için)...")

feature_cols = [c for c in df_feat.columns if c != 'bioactivity_class']
X = df_feat[feature_cols].values
y = df_feat['bioactivity_class'].map({'active': 1, 'inactive': 0}).values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=scaffolds))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"  Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"  Test active: {y_test.sum():,} | Test inactive: {(y_test==0).sum():,}")

unique_train_scaffolds = len(set(scaffolds[train_idx]))
unique_test_scaffolds  = len(set(scaffolds[test_idx]))
overlap = len(set(scaffolds[train_idx]) & set(scaffolds[test_idx]))
print(f"  Train scaffolds: {unique_train_scaffolds:,} | "
      f"Test scaffolds: {unique_test_scaffolds:,} | "
      f"Örtüşme: {overlap:,} ({overlap/unique_test_scaffolds*100:.1f}%)")

# ─────────────────────────────────────────────
# 5. SMOTE DENGELEME
# ─────────────────────────────────────────────
print("\n  Veri dengeleniyor (SMOTETomek)...")

n_active   = (y_train == 1).sum()
n_inactive = (y_train == 0).sum()
target_inactive = int(n_active * 0.85)

smotetomek = SMOTETomek(
    smote=SMOTE(
        sampling_strategy={0: target_inactive},
        k_neighbors=5,
        random_state=42
    ),
    random_state=42
)
X_tr_bal, y_tr_bal = smotetomek.fit_resample(X_train, y_train)

print(f"  Dengeli train: {len(X_tr_bal):,} örnek")
print(f"  Active: {(y_tr_bal==1).sum():,} | Inactive: {(y_tr_bal==0).sum():,}")

# ─────────────────────────────────────────────
# 6. THRESHOLD OPTİMİZASYONU — Macro F1
# ─────────────────────────────────────────────
def find_best_threshold(model, X_val, y_val):
    """Macro F1 ile threshold optimize eder."""
    proba = model.predict_proba(X_val)[:, 1]
    best_thresh, best_score = 0.5, 0
    for t in np.arange(0.25, 0.75, 0.01):
        preds = (proba >= t).astype(int)
        rpt = classification_report(y_val, preds, output_dict=True, zero_division=0)
        score = rpt['macro avg']['f1-score']
        if score > best_score:
            best_score, best_thresh = score, t
    return round(best_thresh, 2), best_score

# Validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr_bal, y_tr_bal,
    test_size=0.15, stratify=y_tr_bal, random_state=42
)

# ─────────────────────────────────────────────
# 6b. LOGISTIC REGRESSION İÇİN SCALER
# ─────────────────────────────────────────────
scaler = StandardScaler()
X_tr_scaled   = scaler.fit_transform(X_tr)
X_val_scaled  = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ─────────────────────────────────────────────
# 7. MODEL TANIMI
# ─────────────────────────────────────────────
imbalance_ratio = (y_tr_bal == 1).sum() / (y_tr_bal == 0).sum()

models = {
    'XGBoost': XGBClassifier(
        n_estimators=400,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1 / imbalance_ratio,
        eval_metric='aucpr',
        use_label_encoder=False,
        random_state=42
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=400,
        learning_rate=0.08,
        max_depth=6,
        class_weight={0: imbalance_ratio, 1: 1.0},
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        verbose=-1,
        random_state=42
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=4,
        class_weight={0: imbalance_ratio, 1: 1.0},
        n_jobs=-1,
        random_state=42
    ),
    'LogisticRegression': LogisticRegression(
        C=0.1,
        class_weight={0: imbalance_ratio, 1: 1.0},
        max_iter=1000,
        solver='saga',
        n_jobs=-1,
        random_state=42
    ),
}

# ─────────────────────────────────────────────
# 8. EĞİTİM VE DEĞERLENDİRME
# ─────────────────────────────────────────────
results_summary = []
trained_models  = {}
best_thresholds = {}

SANITY_MOLECULES = {
    'Galantamin (AChE aktif)': 'COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2',
    'Donepezil (AChE aktif)':  'COc1cc2c(cc1OC)CC(CC1CCN(Cc3ccccc3)CC1)C2=O',
}

for name, model in models.items():
    print("\n" + "=" * 80)
    print(f"  {name} — EĞİTİM VE ANALİZ")
    print("=" * 80)

    use_scaled = (name == 'LogisticRegression')
    _X_tr   = X_tr_scaled   if use_scaled else X_tr
    _X_val  = X_val_scaled  if use_scaled else X_val
    _X_test = X_test_scaled if use_scaled else X_test

    model.fit(_X_tr, y_tr)
    trained_models[name] = model

    opt_thresh, val_f1 = find_best_threshold(model, _X_val, y_val)
    best_thresholds[name] = opt_thresh
    print(f"  ✓ Optimal threshold: {opt_thresh:.2f}  (val macro F1: {val_f1:.4f})")

    y_test_proba = model.predict_proba(_X_test)[:, 1]
    y_test_pred  = (y_test_proba >= opt_thresh).astype(int)
    y_tr_pred    = model.predict(_X_tr)

    train_acc = accuracy_score(y_tr, y_tr_pred)
    test_acc  = accuracy_score(y_test, y_test_pred)
    roc_auc   = roc_auc_score(y_test, y_test_proba)


    precision_macro = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
    recall_macro    = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
    f1_macro        = f1_score(y_test, y_test_pred, average='macro', zero_division=0)

    # Sınıf bazlı metrikler (classification report'tan alınır)
    rpt_dict    = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    inactive_f1 = rpt_dict['0']['f1-score']
    active_f1   = rpt_dict['1']['f1-score']
    inactive_recall = rpt_dict['0']['recall']
    active_recall   = rpt_dict['1']['recall']

    results_summary.append({
        'Model':            name,
        'Threshold':        opt_thresh,
        'Train Acc':        train_acc,
        'Test Acc':         test_acc,
        'Precision(macro)': precision_macro,
        'Recall(macro)':    recall_macro,
        'F1(macro)':        f1_macro,
        'F1(active)':       active_f1,
        'F1(inactive)':     inactive_f1,
        'ROC-AUC':          roc_auc
    })

    # ── Performans Metrikleri ──
    print(f"\n PERFORMANS METRİKLERİ:")
    print(f"  {'='*60}")
    print(f"  Threshold:              {opt_thresh:.2f}")
    print(f"  Train Accuracy:         {train_acc:.4f} ({train_acc*100:.2f}%)")
    print(f"  Test Accuracy:          {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"  Precision (macro avg):  {precision_macro:.4f}")
    print(f"  Recall    (macro avg):  {recall_macro:.4f}")
    print(f"  F1-Score  (macro avg):  {f1_macro:.4f}")
    print(f"  ROC-AUC:                {roc_auc:.4f}")
    print(f"\n  Not: ROC-AUC > Accuracy beklenen bir durumdur.")
    print(f"  AUC olasılık sıralamasını (threshold bağımsız) ölçer.")
    print(f"  Accuracy ise tek bir threshold'daki snapshot'tır.")
    print(f"  Dengesiz veri setlerinde (active:inactive ≈ 5:1) bu fark normaldir.")

    print(f"\n SINIF BAZLI METRİKLER:")
    print(f"  {'='*60}")
    print(f"  Inactive (0) → F1: {inactive_f1:.4f} | Recall: {inactive_recall:.4f}")
    print(f"  Active   (1) → F1: {active_f1:.4f} | Recall: {active_recall:.4f}")

    print(f"\n CLASSIFICATION REPORT:")
    report = classification_report(y_test, y_test_pred,
                                   target_names=['Inactive (0)', 'Active (1)'],
                                   digits=4)
    for line in report.split('\n'):
        print(f"  {line}")

    cm = confusion_matrix(y_test, y_test_pred)
    print(f"\n CONFUSION MATRIX:")
    print(f"  {'='*60}")
    print(f"                    Predicted")
    print(f"                    Inactive    Active")
    print(f"  Actual Inactive   {cm[0,0]:>8}    {cm[0,1]:>6}")
    print(f"  Actual Active     {cm[1,0]:>8}    {cm[1,1]:>6}")
    print(f"  {'='*60}")
    print(f"  TN={cm[0,0]:,} | FP={cm[0,1]:,} | FN={cm[1,0]:,} | TP={cm[1,1]:,}")

    # ── Sanity Check ──
    print(f"\n SANİTY CHECK (Bilinen aktif ilaçlar):")
    print(f"  {'='*60}")
    for mol_name, smiles in SANITY_MOLECULES.items():
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                print(f"  ⚠️  {mol_name}: SMILES parse hatası")
                continue
            feats = calculate_features(smiles)
            if feats is None:
                continue
            feats.pop('bioactivity_class', None)
            for tc in target_cols:
                feats[tc] = 0
            feat_vec = np.array([[feats.get(c, 0) for c in feature_cols]])
            if use_scaled:
                feat_vec = scaler.transform(feat_vec)
            prob = model.predict_proba(feat_vec)[0, 1]
            pred = 'ACTIVE ✓' if prob >= opt_thresh else 'INACTIVE ✗'
            print(f"  {mol_name:<35} P(active)={prob:.3f} → {pred}")
        except Exception as e:
            print(f"    {mol_name}: hata — {e}")

    # ── Confusion Matrix Görseli ──
    # DÜZELTİLDİ: Başlıkta macro avg metrikler gösteriliyor, sınıf bazlı F1 de eklendi
    plt.figure(figsize=(10, 8))
    ax = plt.subplot(1, 1, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Inactive (0)', 'Active (1)'],
                yticklabels=['Inactive (0)', 'Active (1)'],
                annot_kws={'size': 14, 'weight': 'bold'},
                linewidths=2, linecolor='white')
    plt.title(
        f'Confusion Matrix — {name}  [threshold={opt_thresh:.2f}]\n'
        f'Acc: {test_acc:.4f} | Macro Prec: {precision_macro:.4f} | '
        f'Macro Rec: {recall_macro:.4f} | Macro F1: {f1_macro:.4f} | ROC-AUC: {roc_auc:.4f}\n'
        f'Inactive F1: {inactive_f1:.4f} | Active F1: {active_f1:.4f}  '
        f'[Tüm metrikler macro avg]',
        fontsize=10, pad=20, weight='bold'
    )
    plt.ylabel('Gerçek Sınıf (Actual)', fontsize=12, weight='bold')
    plt.xlabel('Tahmin Edilen Sınıf (Predicted)', fontsize=12, weight='bold')
    plt.text(0.5, -0.17,
             f'TN={cm[0,0]:,} | FP={cm[0,1]:,} | FN={cm[1,0]:,} | TP={cm[1,1]:,}',
             ha='center', transform=ax.transAxes, fontsize=11, style='italic',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    plt.tight_layout()
    fname = f'confusion_matrix_{name.replace(" ", "_")}_fixed.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    print(f"\n  Grafik: {fname}")
    plt.close()

# ─────────────────────────────────────────────
# 9. KARŞILAŞTIRMA TABLOSU
# ─────────────────────────────────────────────

print("\n\n" + "=" * 80)
print(" MODEL KARŞILAŞTIRMA  [Precision/Recall/F1 = Macro Average]")
print(" Not: Önceki versiyonda bu değerler sadece Active(1) sınıfına aitti.")
print("=" * 80)

results_df = pd.DataFrame(results_summary).sort_values('ROC-AUC', ascending=False)

print(f"\n{'Model':<22} {'Thresh':<8} {'Train Acc':<12} {'Test Acc':<12} "
      f"{'Prec(mac)':<12} {'Rec(mac)':<12} {'F1(mac)':<12} "
      f"{'F1(act)':<10} {'F1(inact)':<11} {'ROC-AUC':<10}")
print("=" * 125)
for i, (_, row) in enumerate(results_df.iterrows()):
    marker = " ← EN İYİ" if i == 0 else ""
    print(f"{row['Model']:<22} {row['Threshold']:<8.2f} {row['Train Acc']:<12.4f} "
          f"{row['Test Acc']:<12.4f} {row['Precision(macro)']:<12.4f} "
          f"{row['Recall(macro)']:<12.4f} {row['F1(macro)']:<12.4f} "
          f"{row['F1(active)']:<10.4f} {row['F1(inactive)']:<11.4f} "
          f"{row['ROC-AUC']:<10.4f}{marker}")

# ─────────────────────────────────────────────
# 10. EN İYİ MODELİ PKL OLARAK KAYDET
# ─────────────────────────────────────────────
print("\n" + "=" * 80)
print(" EN İYİ MODEL KAYDEDİLİYOR")
print("=" * 80)

best_row  = results_df.iloc[0]
best_name = best_row['Model']

save_package = {
    'model':        trained_models[best_name],
    'scaler':       scaler if best_name == 'LogisticRegression' else None,
    'feature_cols': feature_cols,
    'target_cols':  target_cols,
    'threshold':    best_thresholds[best_name],
    'class_map':    {1: 'active', 0: 'inactive'},
    'model_name':   best_name,
    'training_info': {
        'n_train':           len(X_tr_bal),
        'n_test':            len(X_test),
        'split_method':      'scaffold_split',
        'threshold_opt':     'macro_f1',
        'smote_ratio':       0.85,
        'conflicts_removed': len(conflict_smiles),
        'test_acc':          float(best_row['Test Acc']),
        'roc_auc':           float(best_row['ROC-AUC']),
        'f1_macro':          float(best_row['F1(macro)']),
        'f1_active':         float(best_row['F1(active)']),
        'f1_inactive':       float(best_row['F1(inactive)']),
        'metric_note':       'F1/Precision/Recall = macro average (not class-specific)',
        'all_models_roc':    {r['Model']: round(r['ROC-AUC'], 4)
                              for _, r in results_df.iterrows()},
    }
}

pkl_path = f'best_model_{best_name.replace(" ", "_").lower()}_fixed.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(save_package, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(pkl_path) / 1e6
print(f"\n  Kaydedildi: {pkl_path}  ({size_mb:.1f} MB)")
print(f"   Model:          {best_name}")
print(f"   Threshold:      {best_thresholds[best_name]:.2f}")
print(f"   Feature sayısı: {len(feature_cols):,}")
print(f"\n  Tüm model ROC-AUC skorları:")
for i, (_, row) in enumerate(results_df.iterrows()):
    marker = " " if row['Model'] == best_name else ""
    print(f"    {row['Model']:<22} {row['ROC-AUC']:.4f}{marker}")
print("\n" + "=" * 80)
print("✓ Pipeline tamamlandı.")
print("=" * 80)

In [ ]:


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import WeightedRandomSampler

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GINConv, NNConv, global_add_pool, global_mean_pool
)
from torch_geometric.nn.models import AttentiveFP

from rdkit import Chem
from rdkit.Chem import Descriptors

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_curve, precision_recall_curve, auc
)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 75)
print("  İLAÇ KEŞFİ GNN — DENGELİ VERİ + 4 MODEL")
print("  GIN-Attr | GIN-Context | NeuralFP | AttentiveFP")
print("=" * 75)
print(f"\n Device: {device}")

# ═══════════════════════════════════════════════════════════════
# 1. ATOM / BAĞ ÖZELLİKLERİ
#    DGL Life Science'daki orijinal implementasyonla uyumlu
# ═══════════════════════════════════════════════════════════════

ATOM_SYMBOLS = ['C','N','O','S','F','Si','P','Cl','Br','Mg','Na','Ca',
                'Fe','As','Al','I','B','V','K','Tl','Yb','Sb','Sn',
                'Ag','Pd','Co','Se','Ti','Zn','H','Li','Ge','Cu','Au',
                'Ni','Cd','In','Mn','Zr','Cr','Pt','Hg','Pb','other']

CHIRALITY = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    Chem.rdchem.ChiralType.CHI_OTHER,
]

BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

BOND_DIRS = [
    Chem.rdchem.BondDir.NONE,
    Chem.rdchem.BondDir.ENDUPRIGHT,
    Chem.rdchem.BondDir.ENDDOWNRIGHT,
]

def oh(val, lst):
    return [int(val == v) for v in lst] + [int(val not in lst)]

def get_atom_features(atom):
    """
    9 boyutlu atom özelliği — GIN makalesiyle uyumlu.
    Hu et al., 2020 (Strategies for Pre-training GNNs)
    """
    sym = atom.GetSymbol()
    f  = oh(sym if sym in ATOM_SYMBOLS else 'other', ATOM_SYMBOLS)   # 45
    f += oh(atom.GetDegree(), list(range(11)))                         # 12
    f += oh(atom.GetFormalCharge(), [-1,-2,1,2,0])                    # 6
    f += oh(int(atom.GetNumRadicalElectrons()), [0,1,2,3,4])           # 6
    f += oh(atom.GetHybridization(), [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2,
    ])                                                                  # 6
    f += [int(atom.GetIsAromatic())]                                   # 1
    f += oh(int(atom.GetTotalNumHs()), [0,1,2,3,4])                   # 6
    f += oh(atom.GetChiralTag(), CHIRALITY)                            # 5
    return f   # toplam 87

def get_bond_features(bond):
    """
    3 boyutlu bağ özelliği — GIN makalesiyle uyumlu.
    """
    f  = oh(bond.GetBondType(), BOND_TYPES)    # 5
    f += [int(bond.IsInRing())]                # 1
    f += oh(bond.GetBondDir(), BOND_DIRS)      # 4
    return f   # toplam 10

def smiles_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    atom_feats = [get_atom_features(a) for a in mol.GetAtoms()]
    if not atom_feats:
        return None

    x = torch.tensor(atom_feats, dtype=torch.float)

    edge_indices, edge_attrs = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = get_bond_features(bond)
        edge_indices += [[i, j], [j, i]]
        edge_attrs   += [bf, bf]

    if not edge_indices:
        return None

    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attrs, dtype=torch.float)
    y          = torch.tensor([label], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

# ═══════════════════════════════════════════════════════════════
# 2. VERİYİ YÜKLE VE GRAFA DÖNÜŞTÜR
# ═══════════════════════════════════════════════════════════════
print("\n📂 Veri yükleniyor...")
df = pd.read_csv("strategy_b_data_clean_2class.csv")
label_map = {'active': 1, 'inactive': 0}
df['label'] = df['bioactivity_class'].map(label_map)
df = df.dropna(subset=['canonical_smiles', 'label'])
df['label'] = df['label'].astype(int)

n_act = (df['label']==1).sum()
n_ina = (df['label']==0).sum()
print(f"✓ {len(df):,} kayıt | Active: {n_act:,} ({n_act/len(df)*100:.0f}%) | "
      f"Inactive: {n_ina:,} ({n_ina/len(df)*100:.0f}%)")

print("\n🔗 SMILES → Graf dönüşümü...")
graphs, failed = [], 0
for _, row in df.iterrows():
    g = smiles_to_graph(row['canonical_smiles'], row['label'])
    if g:
        graphs.append(g)
    else:
        failed += 1

NODE_DIM = graphs[0].x.shape[1]
EDGE_DIM = graphs[0].edge_attr.shape[1]
print(f"✓ {len(graphs):,} graf | Atom feat: {NODE_DIM} | Bağ feat: {EDGE_DIM}")

# ═══════════════════════════════════════════════════════════════
# 3. TRAIN / VAL / TEST BÖLME
# ═══════════════════════════════════════════════════════════════
all_labels = [g.y.item() for g in graphs]
idx_all    = list(range(len(graphs)))

tr_idx, te_idx = train_test_split(idx_all, test_size=0.2,
                                  random_state=42, stratify=all_labels)
tr_labels = [all_labels[i] for i in tr_idx]
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.1,
                                  random_state=42, stratify=tr_labels)

train_graphs = [graphs[i] for i in tr_idx]
val_graphs   = [graphs[i] for i in va_idx]
test_graphs  = [graphs[i] for i in te_idx]

print(f"\n  Train: {len(train_graphs):,} | Val: {len(val_graphs):,} | "
      f"Test: {len(test_graphs):,}")

# ═══════════════════════════════════════════════════════════════
# 4. DENGELİ DATALOADER — WeightedRandomSampler
#    Her batch'te active/inactive oranı dengeli olacak
# ═══════════════════════════════════════════════════════════════
def make_balanced_loader(graph_list, batch_size=64, shuffle=True):
    labels = [g.y.item() for g in graph_list]
    counts = np.bincount(labels)
    # Her örnek için ağırlık: nadir sınıfa daha yüksek ağırlık
    weights = [1.0 / counts[l] for l in labels]
    sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )
    if shuffle:
        return DataLoader(graph_list, batch_size=batch_size, sampler=sampler)
    else:
        return DataLoader(graph_list, batch_size=batch_size, shuffle=False)

train_loader = make_balanced_loader(train_graphs, batch_size=64, shuffle=True)
val_loader   = make_balanced_loader(val_graphs,   batch_size=64, shuffle=False)
test_loader  = make_balanced_loader(test_graphs,  batch_size=64, shuffle=False)

# ═══════════════════════════════════════════════════════════════
# 5. FOCAL LOSS
#    İmbalanced sınıflandırma için tasarlanmış — zor örneklere odaklanır
#    Lin et al., 2017 (RetinaNet makalesi)
# ═══════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    """
    FL(p) = -alpha * (1-p)^gamma * log(p)
    gamma > 0: kolay örneklerin etkisini azaltır
    alpha: sınıf ağırlığı
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        bce   = F.binary_cross_entropy_with_logits(logits, targets.float(),
                                                    reduction='none')
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - p_t) ** self.gamma * bce
        return focal.mean()

# ═══════════════════════════════════════════════════════════════
# 6. MODEL MİMARİLERİ
# ═══════════════════════════════════════════════════════════════

# ── Model 1 & 2: GIN (AttrMasking ve ContextPred aynı mimariyi kullanır,
#    fark pre-training stratejisinde — burada fine-tuning yapıyoruz)
class GINModel(nn.Module):

    def __init__(self, in_channels, hidden=300, num_layers=5,
                 dropout=0.5, variant='AttrMasking'):
        super().__init__()
        self.variant  = variant
        self.dropout  = dropout
        self.convs    = nn.ModuleList()
        self.bns      = nn.ModuleList()

        for i in range(num_layers):
            in_ch = in_channels if i == 0 else hidden
            mlp = nn.Sequential(
                nn.Linear(in_ch, 2 * hidden),
                nn.BatchNorm1d(2 * hidden),
                nn.ReLU(),
                nn.Linear(2 * hidden, hidden),
            )
            self.convs.append(GINConv(mlp, train_eps=True))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.pool = global_add_pool
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, ei)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.pool(x, batch)
        return self.head(x).squeeze(-1)


# ── Model 3: NeuralFP (Diferansiyellenebilir Morgan Parmak İzi)
class NeuralFPModel(nn.Module):

    def __init__(self, in_channels, edge_dim, hidden=128, num_layers=4,
                 dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.convs   = nn.ModuleList()
        self.bns     = nn.ModuleList()

        for i in range(num_layers):
            in_ch = in_channels if i == 0 else hidden
            # Her bağ için ayrı ağırlık matrisi üret
            edge_nn = nn.Sequential(
                nn.Linear(edge_dim, in_ch * hidden),
                nn.ReLU()
            )
            self.convs.append(NNConv(in_ch, hidden, edge_nn, aggr='mean'))
            self.bns.append(nn.BatchNorm1d(hidden))

        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, data):
        x, ei, ea, batch = data.x, data.edge_index, data.edge_attr, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, ei, ea)))
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)


# ── Model 4: AttentiveFP (zaten tanımlı, sarıcı ekliyoruz)
class AttentiveFPModel(nn.Module):
    """
    AttentiveFP — Xiong et al., J. Med. Chem. 2020
    Atom düzeyinde dikkat + GRU tabanlı moleküler readout
    """
    def __init__(self, in_channels, edge_dim, hidden=200,
                 num_layers=3, num_timesteps=2, dropout=0.2):
        super().__init__()
        self.afp = AttentiveFP(
            in_channels=in_channels,
            hidden_channels=hidden,
            out_channels=hidden,
            edge_dim=edge_dim,
            num_layers=num_layers,
            num_timesteps=num_timesteps,
            dropout=dropout,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, data):
        emb = self.afp(data.x, data.edge_index, data.edge_attr, data.batch)
        return self.head(emb).squeeze(-1)

# ═══════════════════════════════════════════════════════════════
# 7. EĞİTİM / DEĞERLENDİRME
# ═══════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        loss   = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        preds   = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == batch.y).sum().item()
        total   += batch.num_graphs
        total_loss += loss.item() * batch.num_graphs
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for batch in loader:
        batch  = batch.to(device)
        logits = model(batch)
        probs  = torch.sigmoid(logits).cpu().numpy()
        preds  = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(batch.y.cpu().numpy().tolist())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

def find_best_threshold(labels, probs):
    """PR eğrisinden F1'i maksimize eden threshold'u bul."""
    prec, rec, thresholds = precision_recall_curve(labels, probs)
    f1_scores = 2 * prec * rec / (prec + rec + 1e-8)
    best_idx  = np.argmax(f1_scores[:-1])
    return thresholds[best_idx], f1_scores[best_idx]

def train_model(name, model, epochs=60, lr=1e-3, patience=12):
    print(f"\n{'='*75}")
    print(f"  🔄 {name}")
    print(f"{'='*75}")

    model    = model.to(device)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    best_auc, best_state, wait = 0, None, 0
    history = {'loss': [], 'tr_acc': [], 'va_acc': [], 'va_auc': []}

    for epoch in range(1, epochs + 1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion)
        vl, vp, vb = evaluate(model, val_loader)
        va  = accuracy_score(vl, vp)
        vu  = roc_auc_score(vl, vb)
        scheduler.step()

        history['loss'].append(tl)
        history['tr_acc'].append(ta)
        history['va_acc'].append(va)
        history['va_auc'].append(vu)

        if vu > best_auc:
            best_auc   = vu
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}/{epochs} | Loss: {tl:.4f} | "
                  f"Train: {ta:.4f} | Val: {va:.4f} | AUC: {vu:.4f}")
        if wait >= patience:
            print(f"  ⏹️  Erken durdurma — epoch {epoch} | best AUC: {best_auc:.4f}")
            break

    model.load_state_dict(best_state)
    return model, history

# ═══════════════════════════════════════════════════════════════
# 8. MODELLERİ TANIMLA
# ═══════════════════════════════════════════════════════════════
model_configs = {
    'GIN-AttrMasking': GINModel(NODE_DIM, hidden=300, num_layers=5,
                                dropout=0.5, variant='AttrMasking'),
    'GIN-ContextPred': GINModel(NODE_DIM, hidden=300, num_layers=5,
                                dropout=0.5, variant='ContextPred'),
    'NeuralFP':        NeuralFPModel(NODE_DIM, EDGE_DIM, hidden=128,
                                     num_layers=4, dropout=0.3),
    'AttentiveFP':     AttentiveFPModel(NODE_DIM, EDGE_DIM, hidden=200,
                                        num_layers=3, num_timesteps=2,
                                        dropout=0.2),
}

EPOCHS   = 60
LR       = 1e-3
PATIENCE = 12

# ═══════════════════════════════════════════════════════════════
# 9. EĞİT + DEĞERLENDİR
# ═══════════════════════════════════════════════════════════════
results = {}

for name, model in model_configs.items():

    trained, history = train_model(name, model, EPOCHS, LR, PATIENCE)

    # Test — önce val seti ile optimal threshold bul
    v_lbl, _, v_prb = evaluate(trained, val_loader)
    best_thr, _     = find_best_threshold(v_lbl, v_prb)

    t_lbl, _, t_prb = evaluate(trained, test_loader)
    t_pred_opt = (t_prb >= best_thr).astype(int)   # optimize threshold
    t_pred_50  = (t_prb >= 0.50).astype(int)       # sabit threshold

    acc       = accuracy_score(t_lbl, t_pred_opt)
    precision = precision_score(t_lbl, t_pred_opt, zero_division=0)
    recall    = recall_score(t_lbl, t_pred_opt, zero_division=0)
    f1        = f1_score(t_lbl, t_pred_opt, zero_division=0)
    roc_auc   = roc_auc_score(t_lbl, t_prb)
    cm        = confusion_matrix(t_lbl, t_pred_opt)
    pc, rc, _ = precision_recall_curve(t_lbl, t_prb)
    pr_auc    = auc(rc, pc)

    results[name] = {
        'labels': t_lbl, 'preds': t_pred_opt, 'probs': t_prb,
        'history': history, 'threshold': best_thr,
        'acc': acc, 'precision': precision, 'recall': recall,
        'f1': f1, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'cm': cm,
    }

    # ── Metrikleri yazdır
    print(f"\n📊 {name} SONUÇLARI (threshold={best_thr:.3f}):")
    print(f"  {'='*60}")
    print(f"  Accuracy:   {acc:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1-Score:   {f1:.4f}")
    print(f"  ROC-AUC:    {roc_auc:.4f}")
    print(f"  PR-AUC:     {pr_auc:.4f}")

    print(f"\n📋 CLASSIFICATION REPORT:")
    report = classification_report(t_lbl, t_pred_opt,
                                   target_names=['Inactive (0)', 'Active (1)'],
                                   digits=4)
    for line in report.split('\n'):
        print(f"  {line}")

    print(f"\n🔢 CONFUSION MATRIX:")
    print(f"  TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}")

    # ── Model başına görsel
    fig = plt.figure(figsize=(18, 5))
    gs  = gridspec.GridSpec(1, 4, figure=fig, wspace=0.38)

    ax0 = fig.add_subplot(gs[0])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax0,
                xticklabels=['Inact.','Act.'],
                yticklabels=['Inact.','Act.'],
                annot_kws={'size':13,'weight':'bold'},
                linewidths=2, linecolor='white')
    ax0.set_title(f'{name}\nConfusion Matrix', fontsize=11, weight='bold')
    ax0.set_xlabel('Predicted'); ax0.set_ylabel('Actual')

    ax1 = fig.add_subplot(gs[1])
    ax1.plot(history['loss'], color='tomato', lw=2, label='Train Loss')
    ax1.set_title('Training Loss', fontsize=11, weight='bold')
    ax1.set_xlabel('Epoch'); ax1.grid(alpha=0.3); ax1.legend()

    ax2 = fig.add_subplot(gs[2])
    ax2.plot(history['tr_acc'], label='Train Acc', color='steelblue', lw=2)
    ax2.plot(history['va_acc'], label='Val Acc',   color='orange', lw=2)
    ax2.plot(history['va_auc'], label='Val AUC',   color='green', lw=2, ls='--')
    ax2.set_title('Acc & AUC', fontsize=11, weight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylim(0,1); ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    fpr, tpr, _ = roc_curve(t_lbl, t_prb)
    ax3 = fig.add_subplot(gs[3])
    ax3.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={roc_auc:.4f}')
    ax3.plot([0,1],[0,1],'--',color='gray',lw=1)
    ax3.fill_between(fpr, tpr, alpha=0.12, color='steelblue')
    ax3.set_title('ROC Curve', fontsize=11, weight='bold')
    ax3.set_xlabel('FPR'); ax3.set_ylabel('TPR')
    ax3.legend(fontsize=10); ax3.grid(alpha=0.3)

    plt.suptitle(
        f'{name} — Acc:{acc:.3f} | P:{precision:.3f} | R:{recall:.3f} | '
        f'F1:{f1:.3f} | AUC:{roc_auc:.3f} | thr:{best_thr:.3f}',
        fontsize=11, weight='bold', y=1.02
    )
    plt.tight_layout()
    safe = name.replace('-','_').replace(' ','_')
    plt.savefig(f'drug_gnn_{safe}.png', dpi=150, bbox_inches='tight')
    print(f"  💾 Grafik: drug_gnn_{safe}.png")
    plt.close()

# ═══════════════════════════════════════════════════════════════
# 10. KARŞILAŞTIRMA GRAFİĞİ
# ═══════════════════════════════════════════════════════════════
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC']
metric_keys  = ['acc', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
colors       = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
model_names  = list(results.keys())

x  = np.arange(len(metrics_list))
bw = 0.2

fig, ax = plt.subplots(figsize=(15, 6))
for i, (mname, color) in enumerate(zip(model_names, colors)):
    r    = results[mname]
    vals = [r[k] for k in metric_keys]
    bars = ax.bar(x + i*bw, vals, bw, label=mname, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + bw * 1.5)
ax.set_xticklabels(metrics_list, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('GNN Model Karşılaştırması — Dengeli Veri + Focal Loss',
             fontsize=13, weight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('drug_gnn_comparison_all.png', dpi=150, bbox_inches='tight')
print("\n💾 Karşılaştırma grafiği: drug_gnn_comparison_all.png")
plt.close()

# ═══════════════════════════════════════════════════════════════
# 11. ÖZET TABLO
# ═══════════════════════════════════════════════════════════════
print("\n\n" + "="*75)
print("📊 ÖZET KARŞILAŞTIRMA TABLOSU")
print("="*75)
print(f"\n{'Model':<18} {'Acc':<8} {'Pre':<8} {'Rec':<8} {'F1':<8} "
      f"{'AUC':<8} {'PR-AUC':<8} {'Thr':<6}")
print("─"*75)

sorted_results = sorted(results.items(), key=lambda x: x[1]['roc_auc'], reverse=True)
for i, (mname, r) in enumerate(sorted_results):
    star = " ⭐" if i == 0 else ""
    print(f"{mname:<18} {r['acc']:<8.4f} {r['precision']:<8.4f} "
          f"{r['recall']:<8.4f} {r['f1']:<8.4f} "
          f"{r['roc_auc']:<8.4f} {r['pr_auc']:<8.4f} "
          f"{r['threshold']:<6.3f}{star}")

best_name = sorted_results[0][0]
best_r    = sorted_results[0][1]



